# Lecture 3 — Class Exercise
## Line Charts & Slopegraphs: CO2 Emissions

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv('./data/co2_emissions.csv')
df.head()

## Task 1 — Multi-Series Line Chart with Highlight

In [ ]:
asia=df[df['Region']=='Asia'].copy()
country='China' if 'China' in asia['Country'].unique() else asia.groupby('Country')['CO2_Mt'].max().idxmax()
fig=go.Figure()
for c in asia['Country'].unique():
    d=asia[asia['Country']==c].sort_values('Year')
    if c==country:
        fig.add_trace(go.Scatter(x=d['Year'],y=d['CO2_Mt'],mode='lines',
                                 line=dict(width=4,color='crimson'),name=c))
        fig.add_annotation(x=d['Year'].max(),y=d['CO2_Mt'].iloc[-1],text=c,showarrow=False,xshift=10)
    else:
        fig.add_trace(go.Scatter(x=d['Year'],y=d['CO2_Mt'],mode='lines',
                                 line=dict(width=1,color='lightgray'),
                                 hoverinfo='skip',showlegend=False))
fig.update_layout(template='plotly_white',
title=f'{country} dominates Asia CO2 growth over time',
xaxis_title='Year',yaxis_title='CO2 Emissions (Mt)',
showlegend=False)
fig.show()

## Task 2 — Slopegraph: Regional Change 2000 vs 2022

In [ ]:
reg=df[df['Year'].isin([2000,2022])].groupby(['Region','Year'])['CO2_Mt'].mean().reset_index()
wide=reg.pivot(index='Region',columns='Year',values='CO2_Mt').dropna().reset_index()
wide.columns=['Region','y2000','y2022']
wide['change']=wide['y2022']-wide['y2000']
fig=go.Figure()
for _,r in wide.sort_values('y2022',ascending=False).iterrows():
    color='crimson' if r['change']>0 else 'seagreen'
    fig.add_trace(go.Scatter(x=[0,1],y=[r['y2000'],r['y2022']],mode='lines+markers+text',
        line=dict(color=color,width=3),marker=dict(size=8,color=color),
        text=[f"{r['Region']} {r['y2000']:.0f}",f"{r['Region']} {r['y2022']:.0f}"],
        textposition=['middle left','middle right'],showlegend=False))
fig.update_layout(template='plotly_white',
title='Asia rose sharply while North America declined',
xaxis=dict(tickvals=[0,1],ticktext=['2000','2022'],title='Year'),
yaxis=dict(title='Average CO2 (Mt)',showgrid=False),
margin=dict(l=80,r=140,t=80,b=60))
fig.show()